In [15]:
# ================================
# TRAIN BOTH RNN & LSTM MODELS
# ================================

# Import NumPy → used for numerical operations (arrays, calculations)
import numpy as np

# Import TensorFlow → main deep learning library
import tensorflow as tf

# Import Sequential → used to build neural network layer-by-layer
from tensorflow.keras.models import Sequential

# Import layers:
# LSTM → advanced RNN for long-term dependencies
# SimpleRNN → basic RNN layer
# Dense → output layer (fully connected)
# Embedding → converts words into vectors
# Dropout → helps prevent overfitting
from tensorflow.keras.layers import LSTM, SimpleRNN, Dense, Embedding, Dropout

# Import Tokenizer → converts text into numerical form
from tensorflow.keras.preprocessing.text import Tokenizer

# Import pad_sequences → makes sequences equal length
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Import pickle → used to save/load objects (tokenizer, etc.)
import pickle

In [16]:
# Load data

# Open the file "dataset.txt" (default mode = read)
with open("dataset.txt") as f:
    
    # Read all text from file and convert to lowercase
    # Lowercase helps in consistent processing (no duplicate words like "AI" and "ai")
    data = f.read().lower()

In [17]:
# Tokenizer

# Create tokenizer → converts words into numbers
tokenizer = Tokenizer()

# Learn all unique words from the dataset
# Each word gets a unique index (number)
tokenizer.fit_on_texts([data])

# Total number of unique words
# +1 because index starts from 1 (0 is reserved for padding)
total_words = len(tokenizer.word_index) + 1

In [18]:
# Sequences

# Empty list to store sequences
input_sequences = []

# Loop through each line
for line in data.split("\n"):
    
    # Convert text into numbers
    token_list = tokenizer.texts_to_sequences([line])[0]
    
    # Create small sequences step-by-step
    for i in range(1, len(token_list)):
        
        # Example: [1,2,3] → [1,2], [1,2,3]
        input_sequences.append(token_list[:i+1])


# Find longest sequence length
max_seq_len = max(len(seq) for seq in input_sequences)

# Make all sequences same length (add zeros at start)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')


# Split data

# X → input (all words except last)
X = input_sequences[:, :-1];

# y → output (last word to predict)
# Convert into one-hot (model understands classification)
y = tf.keras.utils.to_categorical(input_sequences[:, -1], num_classes=total_words)

In [19]:
# ================= RNN =================

# Create RNN model
rnn_model = Sequential([
    
    # Embedding → converts words into vectors
    Embedding(total_words, 100, input_length=max_seq_len-1),
    
    # First RNN layer → learns sequence patterns
    # return_sequences=True → passes full sequence to next layer
    SimpleRNN(150, return_sequences=True),
    
    # Second RNN layer → processes final sequence output
    SimpleRNN(100),
    
    # Output layer → predicts next word
    Dense(total_words, activation='softmax')
])

# Compile model
# categorical_crossentropy → for multi-class prediction
# adam → optimizer
rnn_model.compile(loss='categorical_crossentropy', optimizer='adam')

# Train model
# epochs=20 → training cycles
# batch_size=64 → number of samples per step
rnn_model.fit(X, y, epochs=20, batch_size=64)

# Save trained RNN model
rnn_model.save("rnn_model.h5")

Epoch 1/20


C:\Users\Asus\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 1.1781
Epoch 2/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0864
Epoch 3/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0596
Epoch 4/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0534
Epoch 5/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0506
Epoch 6/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0483
Epoch 7/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0484
Epoch 8/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0472
Epoch 9/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0469
Epoch 10/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0470
Epoch 11/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0465
Epoch 12/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0461
Epoch 13/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0467
Epoch 14/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0460
Epoch 15/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0453


In [20]:
# ================= LSTM =================

# Create LSTM model
lstm_model = Sequential([
    
    # Embedding → converts words into vectors (128 dimensions)
    Embedding(total_words, 128, input_length=max_seq_len-1),
    
    # First LSTM layer → learns long-term dependencies
    # return_sequences=True → sends full sequence to next layer
    LSTM(256, return_sequences=True),
    
    # Dropout → randomly disables 20% neurons to prevent overfitting
    Dropout(0.2),
    
    # Second LSTM layer → processes sequence further
    LSTM(128),
    
    # Output layer → predicts next word
    Dense(total_words, activation='softmax')
])

# Compile model
# categorical_crossentropy → for multi-class classification
# adam → optimizer
lstm_model.compile(loss='categorical_crossentropy', optimizer='adam')

# Train model
# epochs=20 → number of training cycles
# batch_size=64 → samples per training step
lstm_model.fit(X, y, epochs=20, batch_size=64)

Epoch 1/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - loss: 2.3992
Epoch 2/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 0.4047
Epoch 3/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 0.1234
Epoch 4/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 0.0781
Epoch 5/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 0.0646
Epoch 6/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 0.0594
Epoch 7/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 0.0554
Epoch 8/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0521
Epoch 9/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 0.0508
Epoch 10/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0497
Epoch 11/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - loss: 0.0491
Epoch 12/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0490
Epoch 13/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0480
Epoch 14/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0481
Epoch 15/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 3s

In [21]:
# Save LSTM model

# Save trained LSTM model to file
lstm_model.save("lstm_model.h5")

# Save tokenizer
# Needed later to convert text into numbers during prediction
pickle.dump(tokenizer, open("tokenizer.pkl", "wb"))

# Save max sequence length
# Required to maintain same input size during testing/generation
pickle.dump(max_seq_len, open("max_seq_len.pkl", "wb"))

# Print confirmation message
print("Training Done!")

Training Done!
